# Plot Condensibility score

##### Project Description:

This is the very first batch of plot. I'm developing base on Sangwoo's data.

##### Working Directory:

/Volumes/BackupHaLab/condense-seq/ipython_notebooks/[Fig.1d]Condensibility_score.2026.ipynb

##### Start Date:

Jan 9, 2026

##### Editor:

Xin Lin

In [1]:
# python modules
import sys, copy, random
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy
from scipy import stats

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Matplotlib:", mpl.__version__)
print("SciPy:", scipy.__version__)

# custom modules
sys.path.append('/Volumes/BackupHaLab/condense-seq/postpro_scripts')
import graphics_edit as graphics
import load_file_edit as load_file
import Interval_dict_python3
import statis_edit as statis

Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:46:49) [Clang 19.1.7 ]
NumPy: 2.2.6
Matplotlib: 3.10.7
SciPy: 1.14.1


In [2]:
# matplotlib setting
%matplotlib inline
mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["axes.facecolor"] = "white"
mpl.rcParams["savefig.facecolor"] = "white"


In [3]:
### parameters
cell_org = {'Exp1':'mouse',
            'Exp2':'mouse',
            'Exp3':'mouse'}

cell_chrnames = {'Exp1':['chr%s' % (i) for i in range(1, 20)] + ['chrX'],
                 'Exp2':['chr%s' % (i) for i in range(1, 20)] + ['chrX'],
                 'Exp3':['chr%s' % (i) for i in range(1, 20)] + ['chrX']}
# DEBUG
print(cell_chrnames)

{'Exp1': ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chrX'], 'Exp2': ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chrX'], 'Exp3': ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chrX']}


In [4]:
### chromosome choices
#chr_choices = cell_chrnames['H1']
chr_choices = ['chr1']

## Loading files

Here are many files loaded into the working space

In [16]:
### load gtab file
working_dir = "/Volumes/BackupHaLab/"
gtab_path = working_dir + "Oct2025_run/analysis_v2/"
# DEBUG
print(gtab_path)

dinfo_dkey = {'Exp1-Samp11_S2_score.gtab.gz':
              {'chr%d_Exp1-Samp11_S2' % (i) :(11, 'Exp1', 'score', i) for i in range(1, 5)}}
# DEBUG
print(dinfo_dkey)

# The following function is NOWHERE to be found...
chr_dkey_ID_value = load_file.read_gtab_batch(dinfo_dkey,
                                               data_path=gtab_path,
                                               chr_choices=chr_choices,
                                               by_chr_first=True)

/Volumes/BackupHaLab/Oct2025_run/analysis_v2/
{'Exp1-Samp11_S2_score.gtab.gz': {'chr1_Exp1-Samp11_S2': (11, 'Exp1', 'score', 1), 'chr2_Exp1-Samp11_S2': (11, 'Exp1', 'score', 2), 'chr3_Exp1-Samp11_S2': (11, 'Exp1', 'score', 3), 'chr4_Exp1-Samp11_S2': (11, 'Exp1', 'score', 4)}}
loading Exp1-Samp11_S2_score.gtab.gz
Done


In [12]:
### read RNA-seq data
# (Jan 9, 2026: Vio replied it's the E14 cell line.)
RNAseq_path = working_dir + 'condense-seq/data/MouseEpigeneticData/E14_RNAseq/'
RNAseq_fname = 'ENCFF827OZU.tsv'
geneID_FPKM = load_file.read_ENCODE_RNA_seq(RNAseq_path + RNAseq_fname)

# DEBUG - confirming the input is valid
print(list(geneID_FPKM.items())[-5:])

[('gSpikein_ERCC-00165', 0.0), ('gSpikein_ERCC-00168', 0.0), ('gSpikein_ERCC-00170', 0.0), ('gSpikein_ERCC-00171', 0.0), ('gSpikein_phiX174', 0.0)]


In [7]:
### figure parameters
# set figure binning parameters
i = 20
bin_size = int(0.5*(10**6) / i) # binsize (unit of bp)
bin_step = bin_size # no overlap
blur_win = int(4*i + 1) # sliding window (unit of bin)

In [14]:
### binning/smoothing the condense-seq data
chr_dkey_sig = {}
for chr in chr_dkey_ID_value:
    for dkey in chr_dkey_ID_value[chr]:
        ID_value = chr_dkey_ID_value[chr][dkey]
        ID_loc = {ID:ID[1:] for ID in ID_value}
        max_pos = genome_size[chr]

        binID_mean = statis.rbin_data_mean(bin_size,
                                           bin_step,
                                           ID_loc,
                                           ID_value,
                                           max_pos=max_pos,
                                           silent=False)

        sig = [binID_mean[binID] for binID in sorted(binID_mean.keys())]
        sig = statis.slow_moving_average2(sig, blur_win)

        if chr not in chr_dkey_sig:
            chr_dkey_sig[chr] = {}
        chr_dkey_sig[chr][dkey] = sig


In [9]:
### rescale/reorganize the RNA-seq data
chr_geneID_pos = {}
chr_geneID_logFPKM = {}
min_FPKM = min(geneID_FPKM.values())
for geneID in geneID_FPKM:
    try:
        chr = geneID_field_value[geneID]['chr']
        pos = geneID_field_value[geneID]['TSS']
    except:
        continue

    if chr not in chr_geneID_pos:
        chr_geneID_pos[chr] = {}
    chr_geneID_pos[chr][geneID] = pos

    logFPKM = np.log2(geneID_FPKM[geneID] - min_FPKM + 1)
    if chr not in chr_geneID_logFPKM:
        chr_geneID_logFPKM[chr] = {}
    chr_geneID_logFPKM[chr][geneID] = logFPKM
    

In [11]:
### binning/smoothing the RNA-seq data
chr_RNA_sig = {}
for chr in chr_choices:
    geneID_pos = chr_geneID_pos[chr]
    geneID_logFPKM = chr_geneID_logFPKM[chr]
    max_pos = genome_size[chr]
    binID_mean = statis.rbin_data_mean(bin_size,
                                       bin_step,
                                       geneID_pos,
                                       geneID_logFPKM,
                                       max_pos=max_pos,
                                       silent=True)

    sig = []
    for binID in sorted(binID_mean.keys()):
        mean = binID_mean[binID]
        if np.isnan(mean):
            mean = 0.0 # put zero expression when no gene found
        sig.append(mean)

    sig = statis.slow_moving_average2(sig, blur_win)
    chr_RNA_sig[chr] = sig


KeyError: 'chr1'

In [ ]:
### make ideogram
chr_Gtype_ideogram = {}
for chr in chr_choices:
    GID_Gband = chr_GID_Gband[chr]
    Gtype_ideogram = {'num':[], 'var':[], 'acen':[]}
    binID_st, binID_ed = 0, genome_size[chr] / bin_step
    for binID in range(binID_st, binID_ed + 1):
        try:
            GID = chr_binID_GID[chr][binID]
            Gtype = GID_Gband[GID]['type']
            Gvalue = GID_Gband[GID]['value']
        except:
            for key in Gtype_ideogram:
                Gtype_ideogram[key].append([np.nan])
            continue

        if Gtype in ['neg', 'pos']:
            Gtype = 'num'
            assert not np.isnan(Gvalue)
        elif Gtype in ['var', 'acen']:
            Gvalue = 10
        else:
            pass
            
        for key in Gtype_ideogram:
            if key == Gtype:
                Gtype_ideogram[key].append([Gvalue])
            else:
                Gtype_ideogram[key].append([np.nan])

    chr_Gtype_ideogram[chr] = Gtype_ideogram

In [ ]:
### set xtick labels along chromosome
chr_xtick_locs = {}
chr_xtick_labels = {}
for chr in chr_choices:
    xtick_locs, xtick_labels = [], []

    binID_st = 0
    binID_ed = genome_size[chr] / bin_step
    for binID in range(binID_st, binID_ed+1):
        pos = bin_step*binID + bin_size/2
        Mb_pos = int(round(float(pos)/(10**6)))

        if Mb_pos % 10 !=0: # 10Mbp steps
            continue

        label = str(Mb_pos)
        if label not in xtick_labels:
            xtick_locs.append(binID)
            xtick_labels.append(label)

    chr_xtick_locs[chr] = xtick_locs
    chr_xtick_labels[chr] = xtick_labels

In [ ]:
### set xtick labels for ideogram
chr_Gtick_locs = {}
chr_Gtick_labels = {}
for chr in chr_GID_binwin:
    GID_binwin = chr_GID_binwin[chr]
    Gtick_locs, Gtick_labels = [], []
    for GID in sorted(GID_binwin.keys()):
        binID_st, binID_ed = GID_binwin[GID]
        pos = (binID_st + binID_ed)/2
        Gname = chr_GID_Gband[chr][GID]['name']
        Gtick_locs.append(pos)
        Gtick_labels.append(Gname)
    chr_Gtick_locs[chr] = Gtick_locs
    chr_Gtick_labels[chr] = Gtick_labels

In [ ]:
### set heterochromatin regions of ideogram
chr_shade_wins = {}
for chr in chr_GID_binwin:
    GID_binwin = chr_GID_binwin[chr]
    shade_wins = []
    for GID in sorted(GID_binwin.keys()):
        Gtype = chr_GID_Gband[chr][GID]['type']
        if Gtype =='pos':
            shade_wins.append(GID_binwin[GID])
    chr_shade_wins[chr] = shade_wins


In [ ]:
### set mask region near centromere
chr_mask_wins = {}
for chr in chr_GID_binwin:
    GID_binwin = chr_GID_binwin[chr]
    mask_wins = []
    for GID in sorted(GID_binwin.keys()):
        Gtype = chr_GID_Gband[chr][GID]['type']
        if Gtype in ['var', 'acen']:
            mask_wins.append(GID_binwin[GID])

    # merge windows near next each others
    mask_wins = sorted(mask_wins)
    new_mask_wins = []
    for win in mask_wins:
        st, ed = win
        if not new_mask_wins:
            new_mask_wins.append((st, ed))
            continue
        prev_st, prev_ed = new_mask_wins.pop()
        if st - prev_ed <= 10:
            new_mask_wins.append((prev_st, max(prev_ed, ed)))
        else:
            new_mask_wins.append((prev_st, prev_ed))
            new_mask_wins.append((st, ed))

    # trim the first part of mask to match the reference genome
    for i in range(len(new_mask_wins)):
        st, ed = new_mask_wins[i]
        new_mask_wins[i] = (st+50, ed)
    
    chr_mask_wins[chr] = new_mask_wins

In [ ]:
### plot genome-wide data along with ideogram
## figure parameters
side_names = {'left':[(1, 'H1', 'score', i) for i in range(1,10)],
              'right':['RNA']}
color_list = np.linspace(0.01, 0.99, num=10)
cmap = mpl.cm.get_cmap("jet")
name_color = {(1, 'H1', 'score', i):cmap(color_list[i]) for i in range(1, 10)}
name_color['RNA'] = 'black'
name_alpha = {'RNA':0.5}
name_label = {(1, 'H1', 'score', i):'[sp]=%.2fmM' % (tnum_conc[i]) for i in range(1, 10)}
name_linestyle = {'RNA':'--'}
side_ylabel={'left':'Condensability fluctuation',
             'right':'Gene expression'}
side_ycolor={'left':'blue',
             'right':'black'}
side_ylim={'left':[-0.5, 0.7],
           'right':None}
#side_ylim={'left':None,
#           'right':None}
#side_yscale={'left':None,
#             'right':None}

## plot data in genome-wide (condensability vs gene expression)
for chr in chr_choices:
    #name_sig = chr_dkey_sig[chr]
    name_sig = {}
    for dkey, sig in chr_dkey_sig[chr].items():
        name_sig[dkey] = statis.standardize (sig, by_std=False)
    name_sig['RNA'] = chr_RNA_sig[chr]
    shade_wins = chr_shade_wins[chr]
    mask_wins = chr_mask_wins[chr]
    graphics.plot_genome_wide(side_names=side_names,
                              name_sig=name_sig,
                              name_color=name_color,
                              name_alpha=name_alpha,
                              name_label=name_label,
                              name_linestyle=name_linestyle,
                              side_ylabel=side_ylabel,
                              side_ycolor=side_ycolor,
                              side_ylim=side_ylim,
                              xtick_locs=chr_xtick_locs[chr],
                              xtick_labels=chr_xtick_labels[chr],
                              Gtype_ideogram=chr_Gtype_ideogram[chr],
                              Gtick_locs=chr_Gtick_locs[chr],
                              Gtick_labels=chr_Gtick_labels[chr],
                              shade_wins=shade_wins,
                              mask_wins=mask_wins,
                              legend_loc='upper left',
                              fig_width=15,
                              fig_height=4,
                              height_ratios=[8, 1],
                              hspace=0.65,
                              save=False)

In [ ]:
### plot genome-wide data along with ideogram
## figure parameters
side_names = {'left':[(1, 'H1', 'score', 8)],
              'right':['RNA']}
name_color = {(1, 'H1', 'score', 8):'tab:blue',
              'RNA':'tab:red'}
name_alpha = {(1, 'H1', 'score', 8):1,
              'RNA':0.5}
name_lw = {(1, 'H1', 'score', 8):2.3,
           'RNA':2}
side_ylabel={'left':'Condensability',
             'right':'Gene expression'}
side_ycolor={'left':'blue',
             'right':'red'}
side_ylim={'left':[1.95, 2.85],
           'right':None}
#side_ylim={'left':None,
#           'right':None}
#side_yscale={'left':None,
#             'right':None}

## plot data in genome-wide (condensability vs gene expression) [Fig.1d]
for chr in chr_choices:
    name_sig = chr_dkey_sig[chr]
    name_sig['RNA'] = chr_RNA_sig[chr]
    shade_wins = chr_shade_wins[chr]
    mask_wins = chr_mask_wins[chr]
    graphics.plot_genome_wide(side_names=side_names,
                              name_sig=name_sig,
                              name_color=name_color,
                              name_alpha=name_alpha,
                              name_lw=name_lw,
                              side_ylabel=side_ylabel,
                              side_ycolor=side_ycolor,
                              side_ylim=side_ylim,
                              xtick_locs=chr_xtick_locs[chr],
                              xtick_labels=chr_xtick_labels[chr],
                              Gtype_ideogram=chr_Gtype_ideogram[chr],
                              Gtick_locs=chr_Gtick_locs[chr],
                              Gtick_labels=chr_Gtick_labels[chr],
                              shade_wins=shade_wins,
                              mask_wins=mask_wins,
                              fig_width=15,
                              fig_height=4,
                              height_ratios=[8, 1],
                              hspace=0.65,
                              # save_path='./data/',
                              save=False,
                              note='H1_NCP_sp')    


In [ ]:
### check the correation between signals (condensability VS gene expression)
for chr in chr_choices:
    X = chr_dkey_sig[chr][(1, 'H1', 'score', 8)]
    Y = chr_RNA_sig[chr]

    fig = plt.figure()
    plt.plot(X, Y, 'k.', alpha=0.15)
    plt.xlabel('Condensability(A.U.)')
    plt.ylabel('Gene expression')
    plt.title ('H1-hESC chr1 (Mb scale)')
    plt.show()

    corr = statis.get_spearman_corr(X, Y)

    print(chr)
    print('Spearman corr:%f' % (corr))

In [ ]:
### plot genome-wide data along with ideogram
## figure parameters
side_names = {'left':[(1, 'H1', 'score', 8)],
              'right':['eigen']}
name_color = {(1, 'H1', 'score', 8):'tab:blue',
              'eigen':'tab:orange'}
name_alpha = {(1, 'H1', 'score', 8):1,
              'eigen':1}
name_lw = {(1, 'H1', 'score', 8):2.3,
           'eigen':2}
side_ylabel={'left':'Condensability',
             'right':'A/B score'}
side_ycolor={'left':'blue',
             'right':'orangered'}
side_ylim={'left':[1.95, 2.85],
           'right':None}
#side_ylim={'left':None,
#           'right':None}
#side_yscale={'left':None,
#             'right':None}

## plot data in genome-wide (condensability vs A/B compartment score) [Extended Data Fig. 4a]
for chr in ['chr1']:
    name_sig = chr_dkey_sig[chr]
    name_sig['eigen'] = chr_eigen_sig[chr]
    shade_wins = chr_shade_wins[chr]
    mask_wins = chr_mask_wins[chr]
    graphics.plot_genome_wide(side_names=side_names,
                              name_sig=name_sig,
                              name_color=name_color,
                              name_alpha=name_alpha,
                              name_lw=name_lw,
                              side_ylabel=side_ylabel,
                              side_ycolor=side_ycolor,
                              side_ylim=side_ylim,
                              xtick_locs=chr_xtick_locs[chr],
                              xtick_labels=chr_xtick_labels[chr],
                              Gtype_ideogram=chr_Gtype_ideogram[chr],
                              Gtick_locs=chr_Gtick_locs[chr],
                              Gtick_labels=chr_Gtick_labels[chr],
                              shade_wins=shade_wins,
                              mask_wins=mask_wins,
                              fig_width=15,
                              fig_height=4,
                              height_ratios=[8, 1],
                              hspace=0.65,
                              save_path='./data/',
                              save=True,
                              note='H1_NCP_sp_ABcompt')
    

In [ ]:
### check the correation between signals (condensability VS A/B compartment score)
for chr in chr_choices:
    X = chr_dkey_sig[chr][(1, 'H1', 'score', 8)]
    Y = chr_eigen_sig[chr]

    fig = plt.figure()
    plt.plot(X, Y, 'k.', alpha=0.1)
    plt.xlabel('Condensability(A.U.)')
    plt.ylabel('A/B score')
    plt.title ('H1-hESC chr1 (Mb scale)')
    plt.show()

    corr = statis.get_spearman_corr(X, Y)

    print(chr)
    print('Spearman corr:%f' % (corr))